# D77 可解释性数据导出(服务器跑一次, json 拷回本地出图)

四个 cell 相互独立可并行(建议 1→4 顺序跑, 3 最耗时 ~1-2h)。跑完把 `artifacts/reports/interpret_*.json` 全部拷回本地。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 日内累计延误差曲线(主图数据): train 最忙日 + 繁忙 val 日 + test 扰动日

In [ ]:
!{PREFIX} scripts/39_interpret_dump.py --mode cumgap --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --days 2023-10-23,2023-11-13,2023-11-27 --tag main


## ② 逐决策 trace: 病理日 11-24 / 悬崖日 11-27 / 技艺日 10-23(train)+ 11-13(val)

In [ ]:
!{PREFIX} scripts/39_interpret_dump.py --mode trace --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --days 2023-11-24,2023-11-27,2023-10-23,2023-11-13 --policies rl,fcfs


## ③ 逐车 leadtime + PPM(train 全 64 天最耗时; val/test 快)

In [ ]:
!{PREFIX} scripts/39_interpret_dump.py --mode leadtime --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split val
!{PREFIX} scripts/39_interpret_dump.py --mode leadtime --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split test
!{PREFIX} scripts/39_interpret_dump.py --mode leadtime --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split train


## ④ 特征组置零消融(val, 验证策略在看什么/fv2 特征是否被使用)

In [ ]:
!{PREFIX} scripts/39_interpret_dump.py --mode permimport --ckpt artifacts/reports/ppo_d73_AB_s0_best.pt --split val
